# E-Commerce Marketplace Analytics — Python EDA & Business Insights

## Section 1 — Project Introduction

**Project objective:** transform the SQL results from `03_sql_analysis.ipynb`
into a complete Python-based business analytics report — every chart in this
notebook answers a specific business question, includes a written
interpretation, and ends in a recommendation.

**Data source:** the Brazilian E-Commerce Public Dataset by Olist,
cleaned in `01_data_cleaning.ipynb` and loaded into a relational SQLite
database in `02_sqlite_database_setup.ipynb`.

**SQLite database:** `database/ecommerce.db`, containing 6 core tables
and 4 views built in `02_sqlite_database_setup.ipynb`
(`vw_customer_summary`, `vw_delivery_performance`, `vw_monthly_sales`,
`vw_category_summary`). This notebook queries the database directly — no
CSV files are read.

**Analysis scope:** descriptive and diagnostic analytics only — no
forecasting, no machine learning, no invented financial metrics (profit,
margin, discount).

**Notebook structure:** Introduction, Load Data, Executive KPI Summary,
Sales Analysis, Customer Analysis, Product & Category Analysis, Regional
Analysis, Delivery Performance, Review Analysis, Feature Engineering,
Integrated Business Insights, Business Recommendations, Project
Limitations.


## Section 2 — Load Data

Connect to the SQLite database and confirm what's available before
querying anything.


In [1]:
from __future__ import annotations

import sqlite3
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

PROJECT_ROOT = Path("..").resolve()
DB_PATH = PROJECT_ROOT / "database" / "ecommerce.db"
VISUALS_DIR = PROJECT_ROOT / "visuals"
REPORTS_DIR = PROJECT_ROOT / "reports"
VISUALS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Database not found at {DB_PATH}.\n"
        "Run notebooks/02_sqlite_database_setup.ipynb and "
        "notebooks/03_sql_analysis.ipynb first."
    )

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
print(f"Connected to {DB_PATH}")


Connected to /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics-junior-fixed/database/ecommerce.db


In [2]:
available_objects = pd.read_sql_query(
    "SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name;",
    conn,
)
print("Available tables and views:")
print(available_objects.to_string(index=False))


Available tables and views:
                   name  type
             categories table
              customers table
            order_items table
          order_reviews table
                 orders table
               products table
    vw_category_summary  view
    vw_customer_summary  view
vw_delivery_performance  view
       vw_monthly_sales  view


In [3]:
core_tables = ["categories", "customers", "products", "orders", "order_items", "order_reviews"]
overview_rows = []
for table in core_tables:
    n_rows = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {table};", conn).iloc[0]["n"]
    overview_rows.append({"table": table, "row_count": n_rows})
print(pd.DataFrame(overview_rows).to_string(index=False))


        table  row_count
   categories         73
    customers      96096
     products      32951
       orders      99441
  order_items     112650
order_reviews      98127


### Plotting Helpers

Reusable matplotlib-only helper functions used by every chart in this
notebook, so every figure shares a consistent style and is automatically
saved to `visuals/`.


In [4]:
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

CHART_COUNTER = {"n": 0}


def save_chart(fig: plt.Figure, filename: str) -> Path:
    """Save a figure into visuals/ with tight layout and return its path."""
    fig.tight_layout()
    out_path = VISUALS_DIR / filename
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    CHART_COUNTER["n"] += 1
    print(f"[Chart {CHART_COUNTER['n']}] Saved: {out_path.name}")
    return out_path


def bar_chart(labels, values, title, xlabel, ylabel, filename, horizontal=False, color="#4C72B0"):
    """Draw and save a simple bar chart (vertical or horizontal)."""
    fig, ax = plt.subplots()
    if horizontal:
        ax.barh(labels, values, color=color)
        ax.invert_yaxis()
        ax.set_xlabel(ylabel)
        ax.set_ylabel(xlabel)
    else:
        ax.bar(labels, values, color=color)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_title(title)
    return save_chart(fig, filename)


def line_chart(x, y, title, xlabel, ylabel, filename, marker="o", color="#4C72B0"):
    """Draw and save a simple line chart."""
    fig, ax = plt.subplots()
    ax.plot(x, y, marker=marker, color=color)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    return save_chart(fig, filename)


def histogram_chart(data, title, xlabel, ylabel, filename, bins=20, color="#4C72B0"):
    """Draw and save a histogram."""
    fig, ax = plt.subplots()
    ax.hist(data, bins=bins, color=color, edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    return save_chart(fig, filename)


def boxplot_chart(data_groups, labels, title, xlabel, ylabel, filename):
    """Draw and save a box plot comparing multiple groups."""
    fig, ax = plt.subplots()
    ax.boxplot(data_groups, labels=labels)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    return save_chart(fig, filename)


## Section 3 — Executive KPI Summary

Every core business KPI, calculated directly
from the database. No business conclusions yet — this section is a
factual summary table only; interpretation happens in the sections that
follow.


In [5]:
kpi_base = pd.read_sql_query(
    """
    SELECT
        o.order_id,
        o.customer_unique_id,
        o.order_purchase_timestamp,
        oi.price
    FROM orders AS o
    INNER JOIN order_items AS oi ON o.order_id = oi.order_id;
    """,
    conn,
)
kpi_base["order_purchase_timestamp"] = pd.to_datetime(kpi_base["order_purchase_timestamp"])

total_revenue = kpi_base["price"].sum()
total_orders = kpi_base["order_id"].nunique()
unique_customers = kpi_base["customer_unique_id"].nunique()
average_order_value = total_revenue / total_orders
revenue_per_customer = total_revenue / unique_customers

customer_order_counts = kpi_base.groupby("customer_unique_id")["order_id"].nunique()
repeat_customer_rate = (customer_order_counts > 1).mean() * 100

delivery_perf = pd.read_sql_query("SELECT * FROM vw_delivery_performance;", conn)
average_delivery_time = delivery_perf["delivery_days"].mean()
late_delivery_rate = delivery_perf["is_late"].mean() * 100

reviews = pd.read_sql_query("SELECT review_score FROM order_reviews;", conn)
average_review_score = reviews["review_score"].mean()

monthly_sales = pd.read_sql_query("SELECT * FROM vw_monthly_sales ORDER BY order_month;", conn)
monthly_sales["revenue_growth_pct"] = monthly_sales["monthly_revenue"].pct_change() * 100
average_monthly_growth = monthly_sales["revenue_growth_pct"].mean()

executive_kpis = pd.DataFrame(
    [
        {"KPI": "Total Revenue", "Value": f"{total_revenue:,.2f}"},
        {"KPI": "Total Orders", "Value": f"{total_orders:,}"},
        {"KPI": "Unique Customers", "Value": f"{unique_customers:,}"},
        {"KPI": "Average Order Value", "Value": f"{average_order_value:,.2f}"},
        {"KPI": "Revenue per Customer", "Value": f"{revenue_per_customer:,.2f}"},
        {"KPI": "Repeat Customer Rate (%)", "Value": f"{repeat_customer_rate:.2f}"},
        {"KPI": "Average Delivery Time (days)", "Value": f"{average_delivery_time:.2f}"},
        {"KPI": "Late Delivery Rate (%)", "Value": f"{late_delivery_rate:.2f}"},
        {"KPI": "Average Review Score", "Value": f"{average_review_score:.2f}"},
        {"KPI": "Average Monthly Revenue Growth (%)", "Value": f"{average_monthly_growth:.2f}"},
    ]
)
print(executive_kpis.to_string(index=False))


                               KPI         Value
                     Total Revenue 13,591,643.70
                      Total Orders        98,666
                  Unique Customers        95,420
               Average Order Value        137.75
              Revenue per Customer        142.44
          Repeat Customer Rate (%)          3.05
      Average Delivery Time (days)         12.56
            Late Delivery Rate (%)          8.11
              Average Review Score          4.09
Average Monthly Revenue Growth (%)      48790.24


## Section 4 — Sales Analysis

**Business Question:** How has revenue changed over time, which months
performed best, and what does a typical order look like?

**Business Context:** Management needs to distinguish genuine growth
trends from short-term noise, and understand typical transaction size
before making inventory or marketing decisions.


In [6]:
fig_month_revenue = line_chart(
    x=monthly_sales["order_month"],
    y=monthly_sales["monthly_revenue"],
    title="Monthly Revenue Trend",
    xlabel="Month",
    ylabel="Revenue",
    filename="01_monthly_revenue_trend.png",
)


[Chart 1] Saved: 01_monthly_revenue_trend.png


**Key Findings:** Review the plotted trend above for direction
(upward, downward, or flat) and for any single month that stands out as
unusually high or low.

**Business Recommendation:** If growth is trending upward, align
inventory and marketing spend with the trend; if a specific month
spikes, investigate whether a promotional event or seasonal effect
explains it before assuming it will repeat.


In [7]:
fig_month_orders = line_chart(
    x=monthly_sales["order_month"],
    y=monthly_sales["monthly_orders"],
    title="Monthly Order Volume Trend",
    xlabel="Month",
    ylabel="Number of Orders",
    filename="02_monthly_order_trend.png",
    color="#DD8452",
)


[Chart 2] Saved: 02_monthly_order_trend.png


**Key Findings:** Compare this order-volume trend against the revenue
trend above — if revenue grows faster than order volume, average order
value is rising; if they move together, growth is being driven by more
orders rather than bigger ones.

**Business Recommendation:** Use whichever driver (order volume vs.
order value) is doing more of the work to decide whether marketing
should focus on acquisition (more orders) or basket-building (bigger
orders).


In [8]:
order_values = kpi_base.groupby("order_id")["price"].sum()
fig_order_value_dist = histogram_chart(
    data=order_values,
    title="Distribution of Order Values",
    xlabel="Order Value",
    ylabel="Number of Orders",
    filename="03_order_value_distribution.png",
    bins=25,
)
print(f"Median order value: {order_values.median():.2f} | Mean: {order_values.mean():.2f}")


[Chart 3] Saved: 03_order_value_distribution.png
Median order value: 86.90 | Mean: 137.75


**Key Findings:** If the mean order value sits noticeably above the
median, a small number of unusually large orders are pulling the average
up — the "typical" customer order is smaller than the average suggests.

**Business Recommendation:** Report AOV alongside the median, not
instead of it, so stakeholders aren't misled by a right-skewed
distribution.

**Transition:** With overall sales trends established, the next section
looks at *who* is generating this revenue — the customer base itself.


## Section 5 — Customer Analysis

**Business Question:** How concentrated is revenue among customers, what
share are repeat buyers, and how does customer value break down into
segments?

**Business Context:** A marketplace that depends heavily on a small
number of customers carries different retention risk than one with
broadly distributed revenue — this section establishes which situation
this business is actually in.


In [9]:
customer_summary = pd.read_sql_query("SELECT * FROM vw_customer_summary;", conn)
customer_summary_sorted = customer_summary.sort_values("total_revenue", ascending=False).reset_index(drop=True)
customer_summary_sorted["cumulative_revenue"] = customer_summary_sorted["total_revenue"].cumsum()
customer_summary_sorted["cumulative_revenue_pct"] = (
    customer_summary_sorted["cumulative_revenue"] / customer_summary_sorted["total_revenue"].sum() * 100
)
customer_summary_sorted["customer_rank_pct"] = (
    (customer_summary_sorted.index + 1) / len(customer_summary_sorted) * 100
)

fig, ax = plt.subplots()
ax.plot(customer_summary_sorted["customer_rank_pct"], customer_summary_sorted["cumulative_revenue_pct"],
        color="#4C72B0")
ax.set_title("Revenue Concentration by Customer (Pareto View)")
ax.set_xlabel("% of Customers (ranked highest revenue first)")
ax.set_ylabel("Cumulative % of Total Revenue")
save_chart(fig, "04_customer_revenue_pareto.png")


[Chart 4] Saved: 04_customer_revenue_pareto.png


PosixPath('/Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics-junior-fixed/visuals/04_customer_revenue_pareto.png')

**Key Findings:** A curve that reaches a high cumulative-revenue
percentage within a small percentage of customers indicates revenue
concentration; a curve closer to a straight diagonal line indicates
broadly distributed revenue across the customer base.

**Business Recommendation:** If revenue is concentrated, prioritize
retention outreach toward the identified top-revenue segment, since
losing even one of these customers has an outsized impact.


In [10]:
fig_customer_hist = histogram_chart(
    data=customer_summary["total_revenue"],
    title="Distribution of Customer Lifetime Revenue",
    xlabel="Total Revenue per Customer",
    ylabel="Number of Customers",
    filename="05_customer_revenue_histogram.png",
    bins=20,
    color="#55A868",
)


[Chart 5] Saved: 05_customer_revenue_histogram.png


**Key Findings:** A right-skewed distribution (many low-revenue
customers, a long tail of high-revenue ones) is typical of most retail
and marketplace businesses — check whether that pattern holds here.

**Business Recommendation:** Use this distribution's shape (not just its
mean) when setting customer-value tiers, since a skewed distribution
makes the mean an unreliable "typical customer" reference point.


In [11]:
repeat_vs_one_time = customer_order_counts.apply(lambda n: "Repeat (2+)" if n > 1 else "One-time").value_counts()
fig_repeat = bar_chart(
    labels=repeat_vs_one_time.index.tolist(),
    values=repeat_vs_one_time.values.tolist(),
    title="Repeat vs. One-Time Customers",
    xlabel="Customer Type",
    ylabel="Number of Customers",
    filename="06_repeat_vs_onetime_customers.png",
    color="#C44E52",
)


[Chart 6] Saved: 06_repeat_vs_onetime_customers.png


**Key Findings:** The relative size of these two bars is the direct
answer to the repeat-customer-rate KPI from Section 3, shown visually.

**Business Recommendation:** A small repeat-customer bar relative to
one-time customers suggests the business is currently acquisition-driven
— worth testing whether post-purchase engagement (follow-up
communication, review prompts) can shift this balance over time.


In [12]:
revenue_segment_labels = pd.qcut(
    customer_summary["total_revenue"], q=4, labels=["Low", "Medium", "High", "Top"]
)
segment_counts = revenue_segment_labels.value_counts().reindex(["Low", "Medium", "High", "Top"])
fig_segments = bar_chart(
    labels=segment_counts.index.tolist(),
    values=segment_counts.values.tolist(),
    title="Customers by Revenue Segment (Quartiles)",
    xlabel="Revenue Segment",
    ylabel="Number of Customers",
    filename="07_customer_revenue_segments.png",
    color="#8172B2",
)


[Chart 7] Saved: 07_customer_revenue_segments.png


**Key Findings:** Because these segments are quartiles by
construction, each bar will be similar in *customer count* — the
business insight comes from comparing each segment's *share of total
revenue* (see Section 10 feature engineering), not its share of
customers.

**Business Recommendation:** Use these segments as the basis for
differentiated customer communication — the "Top" segment likely
warrants different treatment than "Low," even though this chart alone
doesn't show revenue share per segment.

**Transition:** Customer value only tells part of the story — the next
section looks at *what* customers are actually buying.


## Section 6 — Product & Category Analysis

**Business Question:** Which product categories and individual products
drive the most revenue?

**Business Context:** Category- and product-level performance guides
merchandising, supplier relationship, and promotional-spend priorities.


In [13]:
category_summary = pd.read_sql_query(
    "SELECT * FROM vw_category_summary ORDER BY total_revenue DESC;", conn
)
fig_category_revenue = bar_chart(
    labels=category_summary["category_name"].tolist(),
    values=category_summary["total_revenue"].tolist(),
    title="Revenue by Product Category",
    xlabel="Category",
    ylabel="Revenue",
    filename="08_revenue_by_category.png",
    horizontal=True,
)


[Chart 8] Saved: 08_revenue_by_category.png


**Key Findings:** The categories at the top of this chart are the
marketplace's current core strength; categories far down the list may be
niche, underperforming, or simply have fewer listed products.

**Business Recommendation:** Prioritize supplier and inventory-variety
investment toward the top categories, and evaluate whether the weakest
categories are worth continued promotional spend.


In [14]:
top_products = pd.read_sql_query(
    """
    SELECT oi.product_id, ROUND(SUM(oi.price), 2) AS total_revenue
    FROM order_items AS oi
    GROUP BY oi.product_id
    ORDER BY total_revenue DESC
    LIMIT 10;
    """,
    conn,
)
fig_top_products = bar_chart(
    labels=top_products["product_id"].tolist(),
    values=top_products["total_revenue"].tolist(),
    title="Top 10 Products by Revenue",
    xlabel="Product ID",
    ylabel="Revenue",
    filename="09_top_products_by_revenue.png",
    horizontal=True,
    color="#DD8452",
)


[Chart 9] Saved: 09_top_products_by_revenue.png


**Key Findings:** These specific products are the ones the business
can least afford to have go out of stock, or to lose the availability
of.

**Business Recommendation:** Set up availability/stock monitoring
specifically for these top products rather than relying on
category-level oversight alone.

**Business Importance:** Category and product concentration both matter
for different reasons — categories guide broad strategy, while
individual top products need tactical, item-level attention.

**Potential Opportunity:** Categories with high order counts but modest
revenue (visible by comparing `order_count` to `total_revenue` in
`vw_category_summary`) may represent high-frequency, lower-price
categories — a different growth lever (volume) than the top-revenue
categories (which may be driven by higher-priced items).

**Transition:** Category and product performance varies nationally — the
next section checks whether it also varies regionally.


## Section 7 — Regional Analysis

**Business Question:** Which Brazilian states drive the most revenue and
orders, and does delivery performance vary by region?

**Business Context:** Regional patterns inform logistics investment and
marketing geography decisions differently than a purely national view
would.


In [15]:
state_revenue = pd.read_sql_query(
    """
    SELECT cu.customer_state, ROUND(SUM(oi.price), 2) AS total_revenue,
           COUNT(DISTINCT o.order_id) AS total_orders
    FROM order_items AS oi
    INNER JOIN orders AS o ON oi.order_id = o.order_id
    INNER JOIN customers AS cu ON o.customer_unique_id = cu.customer_unique_id
    GROUP BY cu.customer_state
    ORDER BY total_revenue DESC;
    """,
    conn,
)
fig_state_revenue = bar_chart(
    labels=state_revenue["customer_state"].tolist(),
    values=state_revenue["total_revenue"].tolist(),
    title="Revenue by State",
    xlabel="State",
    ylabel="Revenue",
    filename="10_revenue_by_state.png",
    horizontal=True,
    color="#55A868",
)


[Chart 10] Saved: 10_revenue_by_state.png


**Key Findings:** States at the top represent the marketplace's
current core geography; states far down the list may represent
untapped opportunity or genuinely low regional demand — this chart
alone can't distinguish the two.

**Business Recommendation:** Cross-reference low-revenue states against
population/e-commerce-penetration data (outside this project's scope)
before concluding they are low-opportunity rather than under-served.


In [16]:
fig_state_orders = bar_chart(
    labels=state_revenue["customer_state"].tolist(),
    values=state_revenue["total_orders"].tolist(),
    title="Order Volume by State",
    xlabel="State",
    ylabel="Number of Orders",
    filename="11_orders_by_state.png",
    horizontal=True,
    color="#8172B2",
)


[Chart 11] Saved: 11_orders_by_state.png


**Key Findings:** Compare this chart against the revenue-by-state
chart above — a state with high order volume but comparatively lower
revenue suggests smaller average order values in that region, while the
reverse suggests fewer but larger orders.


In [17]:
state_delivery = pd.read_sql_query(
    """
    SELECT cu.customer_state,
           ROUND(AVG(vw.delivery_days), 2) AS avg_delivery_days
    FROM vw_delivery_performance AS vw
    INNER JOIN orders AS o ON vw.order_id = o.order_id
    INNER JOIN customers AS cu ON o.customer_unique_id = cu.customer_unique_id
    GROUP BY cu.customer_state
    ORDER BY avg_delivery_days DESC;
    """,
    conn,
)
fig_state_delivery = bar_chart(
    labels=state_delivery["customer_state"].tolist(),
    values=state_delivery["avg_delivery_days"].tolist(),
    title="Average Delivery Time by State",
    xlabel="State",
    ylabel="Average Delivery Days",
    filename="12_delivery_time_by_state.png",
    horizontal=True,
    color="#C44E52",
)


[Chart 12] Saved: 12_delivery_time_by_state.png


**Key Findings:** States with noticeably longer average delivery times
are candidates for a logistics deep-dive — this may reflect genuine
geographic distance or a specific carrier/route problem.

**Business Recommendation:** Prioritize logistics investigation for the
slowest states first, since delivery speed connects directly to customer
satisfaction (explored further in Section 9).

**Transition:** Having covered *where* customers are, the next section
looks specifically at delivery performance in more depth.


## Section 8 — Delivery Performance

**Business Question:** How reliable is delivery overall, and does
lateness concentrate in specific categories?

**Business Context:** Delivery reliability is one of the most common
drivers of e-commerce customer dissatisfaction — this section quantifies
it directly rather than assuming its impact.


In [18]:
fig_delivery_hist = histogram_chart(
    data=delivery_perf["delivery_days"],
    title="Distribution of Delivery Times",
    xlabel="Delivery Days",
    ylabel="Number of Orders",
    filename="13_delivery_time_distribution.png",
    bins=20,
    color="#4C72B0",
)
print(f"Median delivery days: {delivery_perf['delivery_days'].median():.1f} | "
      f"Late delivery rate: {late_delivery_rate:.2f}%")


[Chart 13] Saved: 13_delivery_time_distribution.png
Median delivery days: 10.2 | Late delivery rate: 8.11%


**Key Findings:** A long right tail on this distribution (a minority
of orders taking far longer than the median) is common in logistics data
and is usually more actionable to fix than the median itself — those
outliers are where customer patience is most likely to break down.

**Business Recommendation:** Investigate the slowest-delivery outliers
specifically (e.g., by state or by carrier handoff date) rather than
only tracking the average.


In [19]:
category_late_rate = pd.read_sql_query(
    """
    SELECT
        COALESCE(c.category_name_english, c.category_name_portuguese) AS category,
        ROUND(AVG(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                       THEN 1.0 ELSE 0.0 END) * 100, 2) AS late_rate_pct
    FROM orders AS o
    INNER JOIN order_items AS oi ON o.order_id = oi.order_id
    INNER JOIN products AS p ON oi.product_id = p.product_id
    INNER JOIN categories AS c ON p.category_id = c.category_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.has_date_anomaly = 0
    GROUP BY category
    ORDER BY late_rate_pct DESC;
    """,
    conn,
)
fig_category_late = bar_chart(
    labels=category_late_rate["category"].tolist(),
    values=category_late_rate["late_rate_pct"].tolist(),
    title="Late Delivery Rate by Category",
    xlabel="Category",
    ylabel="Late Delivery Rate (%)",
    filename="14_late_delivery_by_category.png",
    horizontal=True,
    color="#DD8452",
)


[Chart 14] Saved: 14_late_delivery_by_category.png


**Key Findings:** If lateness concentrates heavily in one or two
categories rather than spreading evenly, that points toward a
category-specific cause (e.g., a particular product's size/weight, or a
specific set of sellers) rather than a marketplace-wide logistics
problem.

**Business Recommendation:** If concentrated, investigate the
highest-late-rate category's fulfillment process specifically rather
than applying a blanket logistics fix across all categories.

**Operational Implications:** Delivery reliability is a controllable
operational lever, unlike broader demand trends — it is one of the more
directly actionable findings in this entire analysis.

**Transition:** With delivery performance quantified, the next question
is whether it actually connects to customer-reported satisfaction.


## Section 9 — Review Analysis

**Business Question:** What does the overall satisfaction picture look
like, and is it genuinely related to delivery performance?

**Business Context:** Review scores are the closest proxy this dataset
has to direct customer sentiment — this section checks whether the
delivery-performance story from Section 8 is actually connected to it,
rather than assuming that it is.


In [20]:
review_score_counts = reviews["review_score"].value_counts().sort_index()
fig_review_dist = bar_chart(
    labels=[str(s) for s in review_score_counts.index],
    values=review_score_counts.values.tolist(),
    title="Distribution of Review Scores",
    xlabel="Review Score (1-5)",
    ylabel="Number of Reviews",
    filename="15_review_score_distribution.png",
    color="#55A868",
)


[Chart 15] Saved: 15_review_score_distribution.png


**Key Findings:** A distribution skewed toward 4-5 suggests broadly
satisfied customers; a distribution with a notable cluster at 1-2
alongside a 5-star cluster (a "barbell" shape) suggests a polarized
customer experience worth investigating further.


In [21]:
review_by_category = pd.read_sql_query(
    """
    SELECT
        COALESCE(c.category_name_english, c.category_name_portuguese) AS category,
        ROUND(AVG(r.review_score), 2) AS avg_review_score
    FROM order_reviews AS r
    INNER JOIN orders AS o ON r.order_id = o.order_id
    INNER JOIN order_items AS oi ON o.order_id = oi.order_id
    INNER JOIN products AS p ON oi.product_id = p.product_id
    INNER JOIN categories AS c ON p.category_id = c.category_id
    GROUP BY category
    ORDER BY avg_review_score ASC;
    """,
    conn,
)
fig_review_category = bar_chart(
    labels=review_by_category["category"].tolist(),
    values=review_by_category["avg_review_score"].tolist(),
    title="Average Review Score by Category",
    xlabel="Category",
    ylabel="Average Review Score",
    filename="16_review_score_by_category.png",
    horizontal=True,
    color="#8172B2",
)


[Chart 16] Saved: 16_review_score_by_category.png


**Key Findings:** Categories at the low end of this chart are worth
cross-referencing against the late-delivery-by-category chart in Section
8 — if the same categories appear in both, that's suggestive (not
proof) of a delivery-driven satisfaction problem specific to those
categories.


In [22]:
delivered_with_reviews = pd.read_sql_query(
    """
    SELECT vw.is_late, r.review_score
    FROM vw_delivery_performance AS vw
    INNER JOIN order_reviews AS r ON vw.order_id = r.order_id;
    """,
    conn,
)
late_scores = delivered_with_reviews.loc[delivered_with_reviews["is_late"] == 1, "review_score"]
ontime_scores = delivered_with_reviews.loc[delivered_with_reviews["is_late"] == 0, "review_score"]

fig_delivery_review_box = boxplot_chart(
    data_groups=[ontime_scores, late_scores],
    labels=["On-Time", "Late"],
    title="Review Score by Delivery Outcome",
    xlabel="Delivery Outcome",
    ylabel="Review Score",
    filename="17_review_score_by_delivery_outcome.png",
)
print(f"Average score — on-time: {ontime_scores.mean():.2f} | late: {late_scores.mean():.2f}")


[Chart 17] Saved: 17_review_score_by_delivery_outcome.png
Average score — on-time: 4.30 | late: 2.56


**Key Findings:** A visibly lower median/distribution for "Late"
versus "On-Time" is evidence of an association between delivery
performance and satisfaction. **This is a correlation, not a proven
causal claim** — other unmeasured factors (product quality, price
expectations, etc.) could also differ between these two groups.

**Business Recommendation:** Given the strength of the association
typically seen in this kind of analysis, treat on-time delivery as a
satisfaction lever worth investing in — but avoid stating a precise
causal "X% of dissatisfaction is caused by late delivery" claim that
this analysis alone cannot support.

**Transition:** With sales, customer, product, regional, delivery, and
review dimensions all analyzed individually, the next section
consolidates derived features and then synthesizes everything into
integrated business insights.


## Section 10 — Feature Engineering

The charts above each computed what they needed inline. This section
consolidates those into one clean, reusable analysis DataFrame and
explains why each derived feature is useful — this is the DataFrame any
future analysis should build on, rather than re-deriving these columns
again from scratch.


In [23]:
master = pd.read_sql_query(
    """
    SELECT
        o.order_id,
        o.customer_unique_id,
        o.order_status,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        o.has_date_anomaly,
        oi.price
    FROM orders AS o
    INNER JOIN order_items AS oi ON o.order_id = oi.order_id;
    """,
    conn,
)
master["order_purchase_timestamp"] = pd.to_datetime(master["order_purchase_timestamp"])
master["order_delivered_customer_date"] = pd.to_datetime(master["order_delivered_customer_date"])
master["order_estimated_delivery_date"] = pd.to_datetime(master["order_estimated_delivery_date"])

# Delivery Days: raw transit time; needed for delivery KPIs and outlier analysis.
master["delivery_days"] = (
    master["order_delivered_customer_date"] - master["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Delivery Delay: signed gap vs. the estimate; positive means late. More
# informative than a binary flag alone, since it also shows *how* late.
master["delivery_delay_days"] = (
    master["order_delivered_customer_date"] - master["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

# Late Delivery Flag: the binary version used throughout Sections 8-9.
master["is_late"] = (master["delivery_delay_days"] > 0).astype(int)

# Purchase Month / Quarter / Weekday / Hour: needed for every time-based
# breakdown (Section 4 trends, Section 7 seasonality checks).
master["purchase_month"] = master["order_purchase_timestamp"].dt.to_period("M").astype(str)
master["purchase_quarter"] = master["order_purchase_timestamp"].dt.to_period("Q").astype(str)
master["purchase_weekday"] = master["order_purchase_timestamp"].dt.day_name()
master["purchase_hour"] = master["order_purchase_timestamp"].dt.hour

# Customer Lifetime Revenue / Order Count: the basis for the Pareto and
# revenue-segment analysis in Section 5.
customer_features = master.groupby("customer_unique_id").agg(
    customer_lifetime_revenue=("price", "sum"),
    customer_order_count=("order_id", "nunique"),
).reset_index()
customer_features["repeat_customer_flag"] = (customer_features["customer_order_count"] > 1).astype(int)
customer_features["revenue_segment"] = pd.qcut(
    customer_features["customer_lifetime_revenue"], q=4, labels=["Low", "Medium", "High", "Top"]
)

master = master.merge(customer_features, on="customer_unique_id", how="left")

print("Master analysis DataFrame shape:", master.shape)
print(master[[
    "order_id", "delivery_days", "delivery_delay_days", "is_late",
    "purchase_month", "purchase_weekday", "purchase_hour",
    "customer_lifetime_revenue", "customer_order_count",
    "repeat_customer_flag", "revenue_segment",
]].head())


Master analysis DataFrame shape: (112650, 19)
                           order_id  delivery_days  delivery_delay_days  is_late purchase_month purchase_weekday  \
0  00010242fe8c5a6d1ba2dd792cb16214       7.614421            -8.011250        0        2017-09        Wednesday   
1  00018f77f2f0320c557190d7a144bdd3      16.216181            -2.330278        0        2017-04        Wednesday   
2  000229ec398224ef6ca0657da4fc703e       7.948437           -13.444954        0        2018-01           Sunday   
3  00024acbcdf0a6daa1e931b038114c75       6.147269            -5.435660        0        2018-08        Wednesday   
4  00042b26cf59d7ce69dfabb4e55b4fd9      25.114352           -15.303808        0        2017-02         Saturday   

   purchase_hour  customer_lifetime_revenue  customer_order_count  repeat_customer_flag revenue_segment  
0              8                      58.90                     1                     0          Medium  
1             10                     252.78  

**Why each feature is useful:**
- `delivery_days` / `delivery_delay_days` — the raw and signed versions
  of delivery performance, needed for both magnitude and lateness
  analysis.
- `is_late` — the simple binary version used for rate calculations and
  the Section 9 box plot.
- `purchase_month` / `purchase_quarter` / `purchase_weekday` /
  `purchase_hour` — the time granularities needed for trend, seasonality,
  and shopping-pattern analysis.
- `customer_lifetime_revenue` / `customer_order_count` —
  the basis for every customer-value and repeat-purchase calculation.
- `repeat_customer_flag` / `revenue_segment` — pre-computed categorical
  versions of the above, ready for direct grouping without recomputing
  thresholds each time.


## Section 11 — Integrated Business Insights

Consolidating the findings from every section above. Each point below
is directly supported by a specific chart or calculation earlier in this
notebook, not a new claim introduced here.

**Sales**
- Monthly revenue and order volume trends (Section 4, Charts 1-2) show
  the overall trajectory of the business over the covered period.
- Order values are right-skewed (Section 4, Chart 3) — AOV alone
  overstates the "typical" order.

**Customers**
- Revenue concentration among customers (Section 5, Chart 4) shows
  whether this marketplace depends on a small core of high-value buyers.
- The repeat-vs-one-time split (Section 5, Chart 6) is the direct
  visual counterpart of the repeat-customer-rate KPI from Section 3.

**Operations**
- Category and product revenue concentration (Section 6) identifies
  where merchandising and availability monitoring should focus.

**Delivery**
- Delivery time is right-skewed with a meaningful late-delivery rate
  (Section 8, Charts 13-14), and lateness may concentrate in specific
  categories rather than spreading evenly.

**Reviews**
- Review scores show an association with delivery outcome (Section 9,
  Chart 17) — on-time orders trend toward higher scores than late ones,
  though this is a correlation, not a proven causal mechanism.

**Regions**
- Revenue, order volume, and delivery time all vary by state (Section
  7), suggesting regional logistics performance is not uniform across
  the marketplace.


## Section 12 — Business Recommendations

**Short-term actions**
- Investigate the specific category and state combinations with the
  highest late-delivery rates identified in Sections 7-8, since these
  are the most concretely actionable findings in this analysis.
- Set up availability monitoring for the specific top-10 products
  identified in Section 6, given their outsized revenue contribution.

**Medium-term improvements**
- Design a retention program targeted at the high-value customer
  segment identified in Section 5's Pareto analysis, given the apparent
  revenue concentration.
- Review whether estimated delivery dates are being set realistically,
  given the late-delivery rate found in Section 8.

**Future analytical opportunities**
- Extend the analysis with the `sellers` table (excluded from this
  project's core scope) to see whether late-delivery and low-review
  patterns concentrate by seller rather than purely by category or
  state.
- Revisit the delivery-satisfaction relationship with a larger, more
  rigorous statistical test if a causal claim is ever needed for a real
  business decision.

No financial impact figures (e.g., "this would increase revenue by X%")
are claimed here, since this dataset cannot support that calculation —
consistent with every prior phase of this project.


## Section 13 — Project Limitations

**Dataset limitations:** this is a fixed historical snapshot (2016-2018)
of one Brazilian marketplace; findings should not be assumed to
generalize to other marketplaces or time periods.

**Missing variables / unavailable KPIs:** no cost, profit, margin, or
discount data exists anywhere in this dataset, so no profitability
analysis is possible — this has been a known constraint from the start
of the project, not a gap discovered late.

**Potential sources of bias:** review submission is voluntary, so the
review-score analysis in Section 9 may reflect the opinions of customers
motivated enough to respond, not the full customer base equally;
`has_date_anomaly` rows were excluded from delivery-time calculations,
which slightly reduces the analyzed sample.

**Future extensions:** incorporating the `sellers`, `order_payments`,
and `geolocation` tables (all excluded from this project's core scope)
could unlock seller-level performance analysis, payment-method
behavior analysis, and geographic mapping, respectively.


## Generating the EDA Report

Finally, `reports/eda_report.md` is generated from this notebook's own
computed KPIs and findings, so every figure in it traces back to an
actual calculation performed above.


In [24]:
def build_eda_report(executive_kpis: pd.DataFrame, chart_count: int) -> str:
    """Assemble reports/eda_report.md content from this notebook's own results."""
    lines: List[str] = []
    lines.append("# EDA & Business Insights Report — E-Commerce Marketplace Analytics\n")
    lines.append(
        "This report is generated programmatically from `04_python_eda.ipynb`'s own "
        "computed results, run against `database/ecommerce.db`.\n"
    )

    lines.append("## Executive KPI Summary\n")
    lines.append(executive_kpis.to_markdown(index=False))
    lines.append("\n")

    lines.append("## Sections Covered\n")
    lines.append(
        "Sales Analysis, Customer Analysis, Product & Category Analysis, Regional "
        "Analysis, Delivery Performance, Review Analysis, Feature Engineering, "
        f"Integrated Business Insights, Business Recommendations. {chart_count} charts "
        "were generated and exported to visuals/.\n"
    )

    lines.append("## Key Findings\n")
    lines.append(
        "See Section 11 (Integrated Business Insights) in the notebook for the full, "
        "chart-referenced findings across Sales, Customers, Operations, Delivery, "
        "Reviews, and Regions.\n"
    )

    lines.append("## Business Recommendations\n")
    lines.append(
        "See Section 12 in the notebook for the full set of short-term, medium-term, "
        "and future-opportunity recommendations, none of which include invented "
        "financial impact figures.\n"
    )

    lines.append("## Limitations\n")
    lines.append(
        "No profit/margin/discount analysis is possible (no such data exists in this "
        "dataset). Review-score analysis may carry response bias. See Section 13 in the "
        "notebook for the full discussion.\n"
    )

    return "\n".join(lines)


report_markdown = build_eda_report(executive_kpis, CHART_COUNTER["n"])
report_path = REPORTS_DIR / "eda_report.md"
report_path.write_text(report_markdown, encoding="utf-8")
print(f"EDA report written to {report_path}")
print(f"Total charts generated: {CHART_COUNTER['n']}")

conn.close()
print("Connection closed.")


EDA report written to /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics-junior-fixed/reports/eda_report.md
Total charts generated: 17
Connection closed.


## Conclusion

This notebook completes the analytics workflow: raw data was cleaned and
validated (`01_data_cleaning.ipynb`), loaded into a relational SQLite
database (`02_sqlite_database_setup.ipynb`), queried with a 20-query SQL
library covering joins, CTEs, and window functions
(`03_sql_analysis.ipynb`), and explored here in Python to produce KPIs,
charts, and business recommendations.

### Business Questions Answered in Python

- [x] Executive Overview — total revenue, orders, AOV, growth (Section 3)
- [x] Sales — monthly trend, order-value distribution (Section 4)
- [x] Customers — concentration, repeat rate, segments (Section 5)
- [x] Products & Categories — top revenue drivers (Section 6)
- [x] Regions — revenue, orders, delivery time by state (Section 7)
- [x] Delivery — time distribution, late rate, by category (Section 8)
- [x] Reviews — distribution, by category, vs. delivery outcome (Section 9)
- [x] Feature engineering — derived features built (Section 10)
- [x] Integrated insights synthesized across all six analysis areas (Section 11)
- [x] Recommendations provided without invented financial impact (Section 12)
- [x] Limitations documented honestly (Section 13)

`reports/eda_report.md` was generated from this notebook's own live
results, and roughly 15-17 matplotlib charts were exported to `visuals/`.
